In [1]:
import sys
sys.path.append("/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages")

In [17]:
import numpy as np
import os
import datetime as dt
from squaternion import Quaternion
import pickle

---

**Quick experiment**

In [3]:
ORIENTATION_NOISE = 0.01

In [4]:
# our random linear transform
a = np.random.normal(0, 0.1, size=(11, 256))

In [5]:
def generate() -> np.array:
    dim = np.random.uniform(0.1, 0.5, size=3).round(4)
    pos = [*np.random.uniform(-3.0, 3.0, size=2).round(4), 
           max(np.round(np.random.uniform(0, 3.0), 4), dim[2]/2+0.01)]
    q = np.array(Quaternion.from_euler(0., 0., 0.))
    # add a little bit of noise to the orientation
    noise = np.random.normal(0, ORIENTATION_NOISE, size=4)
    q = (q + noise).round(4)
    orn = q[[1, 2, 3, 0]]  # re-order so it's the same ordering as in pybullet 
    m = np.prod(dim).round(4)*10  
    return np.array([*pos, *dim, *orn, m])

In [6]:
# example
v = generate()
v

array([-2.518 , -0.6606,  2.8112,  0.1028,  0.4332,  0.4397, -0.011 ,
        0.0259, -0.0121,  1.0007,  0.196 ])

In [7]:
# randomly linearly projected vector of x -> embedding
x = v @ a
x

array([ 0.29304411,  0.28120959,  0.34439032, -0.25127191,  0.30680969,
        0.45296791, -0.59946609,  0.17686071,  0.53105682, -0.13053459,
       -0.26382695,  0.06506497,  0.13014926, -0.20984114, -0.41866007,
       -0.12296642, -0.05504751, -0.26948286,  0.16509869, -0.36128617,
        0.2007357 , -0.1940393 ,  0.3752284 ,  0.24040842,  0.00183909,
        0.09372463,  0.07366902, -0.33388524, -0.1406355 ,  0.43041778,
        0.45767887, -0.21210133,  0.38010322,  0.35103036,  0.85521421,
       -0.36951315,  0.11133407,  0.64832428,  0.36752871, -0.14613992,
        0.13043196,  0.22825197, -0.22591842, -0.79868244,  1.18164857,
       -0.02831334, -0.04404305,  0.2628362 ,  0.15575517, -0.23038935,
       -0.15811815, -1.08748069,  0.19818905,  0.34863836, -0.5695733 ,
       -0.1056094 ,  0.02797144, -0.40362199, -0.16355938,  0.37568108,
       -0.54401474, -0.21659687, -0.01531301,  0.60894912,  0.66036932,
       -0.34211695,  0.42782271,  0.21957153,  0.75128223,  0.19

---

**Creating an embedding dataset from the simulations**

In [ ]:
NUM_SAMPLES_PER_TRAJECTORY = 10
SPLIT = 0.7

In [ ]:
num_simulations = len([filename for filename in os.listdir("cuboid_simulations") if ".npy" in filename])
num_train_simulations = int(SPLIT*num_simulations)
print(num_simulations, "total simulations")
print(num_train_simulations, "training simulations")

1000 total simulations
700 training simulations


In [ ]:
V = []
for filename in os.listdir("cuboid_simulations"):
    if ".npy" in filename:
        v = np.load(open("cuboid_simulations/" + filename, "rb"))
        # 500+ samples for one item's trajectory in one particular simulation is probably too dense
        # I will sample NUM_SAMPLES_PER_TRAJECTORY entries, and try to have enough simulations and items 
        # to have a large and diverse dataset
        v = v[np.random.choice(range(len(v)), size=min(NUM_SAMPLES_PER_TRAJECTORY, len(v)), replace=False)]
        # Here I don't care about different timesteps. I simply want a list of 11-vectors
        # v = v.reshape(-1, 11)
        V.append(v)

V_train = np.concatenate(V[:num_train_simulations])
V_test = np.concatenate(V[num_train_simulations:]) 

# Normalize the data using statistics from the training set
mean, std = V_train.mean(), V_train.std()
V_train = (V_train-mean)/std
V_test = (V_test-mean)/std

# No longer needed
del V, mean, std

In [ ]:
# Store the whole dataset (before applying embeddings since I'm just using a random linear projection here for experimentation) 
# as a single pickled file

timestamp = str(dt.datetime.now()).replace(" ", "_").replace(":", "_").replace("-", "_").replace(".", "_")
pickle.dump((V_train, V_test), open(f"datasets/embeddings/dataset_{timestamp}.pickle", "wb"))

# E.g. in Colab, read it like this:
# V_train, V_test = pickle.load(open("datasets/embeddings/dataset_2023_01_18_16_47_48_330440.pickle", "rb"))

In [ ]:
"""
# If I didn't merge the timestep and item dimensions:
# Apply embedding transform to each item in each timestep
# Notice that (V_train @ a)[0,0] == V_train[0,0] @ a, so I can just do (V_train @ a) which is more efficient
X_train = (V_train @ a)
X_test = (V_test @ a)
"""

# Apply embedding transform to each vector 
X_train = (V_train @ a)
X_test = (V_test @ a)
(X_train.shape[0], X_test.shape[0])

---

**Principal Component Analysis**

Just for fun, what would happen if I applied PCA to these embeddings?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
pca = PCA(n_components=2).fit(X_train)
# looks similar to if I applied PCA directly to the raw vectors, V_train

In [ ]:
Y_train = pca.transform(X_train)
Y_test = pca.transform(X_test)

In [ ]:
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.scatter(Y_train[:,0], Y_train[:,1], alpha=0.1)

In [ ]:
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.scatter(Y_test[:,0], Y_test[:,1], alpha=0.1)

In [ ]:
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.scatter(Y_train[:,0], Y_train[:,1], alpha=0.1)
plt.scatter(Y_test[:,0],  Y_test[:,1],  alpha=0.1)

In [ ]:
Y_test